<a href="https://colab.research.google.com/github/Priyaa1904/Flyrank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyaa1904/Flyrank-ML-Internship-Starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature vector

For the Content Refresh lane, one row represents one content page observed at a decision point.

I use five numerical features:

- `gsc_impressions` — historical search visibility.
- `gsc_avg_position` — historical average search position.
- `gsc_clicks` — historical search clicks.
- `word_count` — content length.
- `search_volume` — search demand associated with the content.

The first three come from the daily performance table, while `word_count` and `search_volume` come from `dim_content`. I keep identifiers such as `client_hash_id` and `content_hash_id` as context rather than model features.

In [16]:
%pip -q install duckdb

In [17]:
import duckdb

con = duckdb.connect()

REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

content_rel = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
)
"""

print("Warehouse connection ready.")

Warehouse connection ready.


In [18]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [19]:
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [20]:
# Build the five-feature vector from the March 2026 partition

feature_vector = con.sql(f"""
    SELECT
        p.client_hash_id,
        p.content_hash_id,
        p.gsc_impressions,
        p.gsc_avg_position,
        p.gsc_clicks,
        c.word_count,
        c.search_volume
    FROM {REL} AS p
    LEFT JOIN {content_rel} AS c
        ON p.client_hash_id = c.client_hash_id
        AND p.content_hash_id = c.content_hash_id
    WHERE p.gsc_data_available IS TRUE
    LIMIT 1000
""").df()

print("Feature vector shape:", feature_vector.shape)
print("\nFeature columns:")
print(feature_vector.columns.tolist())

feature_vector.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (1000, 7)

Feature columns:
['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_avg_position', 'gsc_clicks', 'word_count', 'search_volume']


,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,gsc_clicks,word_count,search_volume
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,3.350000,0,<NA>,20
1,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0.000000,0,<NA>,10
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,4.928000,1,2123,20
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,4.000000,0,<NA>,90
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,2.272727,0,<NA>,40


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

| Feature | Meaning | Missing-value handling | Available when? |
|---|---|---|---|
| `gsc_impressions` | Search impressions for the content page | Keep as numeric; handle missingness explicitly if needed | Available before the decision because it comes from historical Search Console performance |
| `gsc_avg_position` | Average search position for the content page | Treat `0` as no search-position data, not as a real rank | Available before the decision because it comes from historical Search Console performance |
| `gsc_clicks` | Search clicks for the content page | Keep as numeric; handle missingness explicitly if needed | Available before the decision because it comes from historical Search Console performance |
| `word_count` | Number of words in the content | Preserve missingness and add a missing-value indicator rather than blindly filling with 0 | Available before the decision because it is a stored content attribute |
| `search_volume` | Search demand associated with the content | Preserve missingness if present and handle it explicitly | Available before the decision because it is a stored search-demand attribute |

The two hash IDs are retained only as context for grouping, joining, and validation. They are not model features.

In [21]:
# Check missingness in the five model features

model_features = [
    "gsc_impressions",
    "gsc_avg_position",
    "gsc_clicks",
    "word_count",
    "search_volume"
]

missing_summary = (
    feature_vector[model_features]
    .isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_pct"] = (
    missing_summary["missing_count"]
    / len(feature_vector)
    * 100
)

missing_summary

,missing_count,missing_pct
gsc_impressions,0,0.0
gsc_avg_position,0,0.0
gsc_clicks,0,0.0
word_count,657,65.7
search_volume,3,0.3


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage checks

The feature vector must contain only information available at the decision moment.

I specifically check for three categories of leakage:

1. **Label-derived information** — fields used to construct the target/proxy must not be features.
2. **Future-window information** — information from after the March decision point must not be included.
3. **Product-decision flags** — operational labels or flags created from the outcome should not be used as model inputs.

For the Content Refresh lane, `trend_direction`, `trend_pct`, and `is_declining_label` are excluded because they are derived from the declining/outcome signal. April performance is also excluded when the decision point is March because it belongs to the future outcome window.

In [22]:
# Actual leakage/exclusion check

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "april_impressions"
]

present_forbidden = [
    col for col in forbidden_features
    if col in feature_vector.columns
]

print("Forbidden/leakage-prone columns present in feature vector:")
print(present_forbidden)

assert len(present_forbidden) == 0

print("\nLeakage check passed: no forbidden columns are in the feature vector.")

Forbidden/leakage-prone columns present in feature vector:
[]

Leakage check passed: no forbidden columns are in the feature vector.


In [23]:
# Verify that future April data is not present in the March feature vector

future_columns = [
    "april_impressions",
    "april_clicks",
    "april_sessions",
    "future_impressions",
    "future_clicks"
]

future_present = [
    col for col in future_columns
    if col in feature_vector.columns
]

print("Future-window columns present:")
print(future_present)

assert len(future_present) == 0

print("\nFuture-window leakage check passed.")

Future-window columns present:
[]

Future-window leakage check passed.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields

I deliberately exclude the following from the feature vector:

- `client_hash_id` and `content_hash_id` — identifiers used for joins, grouping, and validation, not predictive features.
- `trend_direction` and `trend_pct` — label-derived trend information and therefore unsafe as model inputs.
- `is_declining_label` — directly represents the outcome and would leak the target.
- Future-period performance such as April metrics — unavailable at the March decision moment.
- Product-decision flags or operational labels — these represent decisions or outcomes rather than information available for prediction.

These exclusions keep the feature vector restricted to information that could genuinely be known when the Content Refresh decision is made.

In [24]:
# Final feature-vector audit

final_features = [
    "gsc_impressions",
    "gsc_avg_position",
    "gsc_clicks",
    "word_count",
    "search_volume"
]

print("Final model features:")
print(final_features)

print("\nNumber of model features:", len(final_features))

assert len(final_features) <= 5
assert all(col in feature_vector.columns for col in final_features)

print("\nFinal feature audit passed.")

Final model features:
['gsc_impressions', 'gsc_avg_position', 'gsc_clicks', 'word_count', 'search_volume']

Number of model features: 5

Final feature audit passed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.